# MarketPulse — Data Ingestion & Initial Exploration

**Goal of this notebook:**
Load all raw Olist e-commerce tables, inspect their shape, data types, and missing values, and document findings that will drive the cleaning rules used in the PySpark ETL notebook (`02_spark_etl.ipynb`).

This notebook does **not** clean the data — it only explores and documents. All cleaning logic lives downstream in the ETL step, so this stays a clear record of the raw data's condition.

## 1. Imports

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 2. Load Raw Tables

All raw CSVs live in `data/raw/` (not committed to git — see `README.md` for download instructions from Kaggle: *Brazilian E-Commerce Public Dataset by Olist*).

In [2]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

tables = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'payments': payments,
    'reviews': reviews,
}

## 3. Shape, Dtypes, and Null Check — All Tables

For every table: row/column count, data types, and count of missing values per column.

In [4]:
for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"{name.upper()}  —  shape: {df.shape}")
    print(f"{'='*60}")
    print("\nDtypes:")
    print(df.dtypes)
    print("\nNull counts:")
    print(df.isnull().sum())


CUSTOMERS  —  shape: (99441, 5)

Dtypes:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Null counts:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

ORDERS  —  shape: (99441, 8)

Dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Null counts:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered

## 4. Order Status Breakdown

Understanding how many orders are `delivered` vs `canceled` vs other statuses matters for later revenue and retention logic — only completed orders should typically count toward revenue metrics.

In [5]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 5. Findings Summary

| Table | Rows | Key Finding | Action Needed |
|---|---|---|---|
| customers | 99,441 | No nulls | None |
| orders | 99,441 | Date columns are strings, not datetime. Nulls in `order_approved_at` (160), `order_delivered_carrier_date` (1,783), `order_delivered_customer_date` (2,965) | Convert to datetime in ETL. Nulls are expected — cancelled/in-transit orders won't have these dates. Do **not** fill; create a derived `is_delivered` flag instead. |
| order_items | 112,650 | More rows than orders (one order can have multiple items). No nulls | None |
| products | 32,951 | 610 nulls in `product_category_name` and descriptive fields (~1.9%). 2 nulls in dimension fields | Label missing category as `"unknown"` rather than drop (avoids silently dropping revenue in aggregations). Drop the 2 rows missing dimensions — trivial. |
| payments | 103,886 | More rows than orders (multiple payment installments/methods per order). No nulls | None |
| reviews | 99,224 | Heavy nulls in `review_comment_title` (87,656) and `review_comment_message` (58,247) | Expected — most customers rate without writing text. No cleaning needed since only `review_score` is used numerically; comment text is out of scope for this project. |
| orders (status) | — | 97.0% `delivered` (96,478). Remaining ~3% split across `shipped` (1,107), `canceled` (625), `unavailable` (609), `invoiced` (314), `processing` (301), `created` (5), `approved` (2) | Filter to `order_status == 'delivered'` for revenue and RFM feature calculations. Keep all statuses for order-volume and cancellation-rate analysis as a separate metric. |

**Next step:** these findings define the cleaning rules implemented in `02_spark_etl.ipynb`.